In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("Use catalog ekart_dt")
silver_run_id = str(uuid.uuid4)
print("current silver run id :",silver_run_id)

# Silve control Table

In [0]:
spark.sql("""
          create table if not exists ekart_dt.silver.processing_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at string,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          """)

# Helper Function
### Reusable Logic for Silver

This cell contains reusable logic for Silver:

- `upsert_to_silver()` merges cleaned / transformed rows into the Silver target table
- `get_last_processed_bronze_ingested_at()` reads the Silver watermark
- `upsert_silver_control()` updates the Silver control table
- `get_incremental_bronze()` reads only new Bronze rows that Silver has not processed yet


In [0]:
#upsert_to_silver function
def upsert_to_silver(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark,target_table)
        (dt.alias("t").merge(df_source.alias("s"),f"t.{join_key}=s.{join_key}")        
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)


In [0]:
#get_last_processed_bronze_ingested_at() reads the Silver watermark
def get_last_processed_bronze_ingested_at(entity_name: str):
    ctrl = (
        spark.table("ekart_dt.silver.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None

    return rows[0]["last_processed_bronze_ingested_at"]

In [0]:
#upsert_silver_control() updates the Silver control table

def upsert_silver_control(
    entity_name,
    last_processed_bronze_run_id,
    last_processed_bronze_ingested_at,
    rows_merged
):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "SUCCESS",
                silver_run_id,
                datetime.utcnow(),
            )
        ],
        schema="""
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,
        silver_run_id string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(
        spark,
        "datatocrunch_novacart_adb.silver_schema.processing_control"
    )

    (
        dt.alias("t")
        .merge(
            ctrl_df.alias("s"),
            "t.layer = s.layer and t.entity_name = s.entity_name"
        )
        .whenMatchedUpdate(
            set={
                "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
                "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
                "rows_merged": "s.rows_merged",
                "run_status": "s.run_status",
                "silver_run_id": "s.silver_run_id",
                "updated_at": "s.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
#get_incremental_bronze() reads only new Bronze rows that Silver has not processed yet

def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)

    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at

    return (
        bronze_df.filter(
            F.col("bronze_ingested_at") > F.lit(last_ingested_at)
        ),
        last_ingested_at
    )

## Step 4 — Orders Incremental Processing

This cell processes **orders** from Bronze to Silver.

It does the following:

- reads only new Bronze order rows
- cleans values like `order_status` and `order_amount`
- keeps only the latest version per `order_id`
- validates business rules
- sends bad rows to quarantine
- merges good rows into `orders_transformed`

In [0]:
# Count the incremental order rows entering Silver in this run.
orders_inc_count = orders_inc.count()
print(f"orders_rows_to_process_in_silver = {orders_inc_count}")

# Only run Silver order cleaning and validation when there are new Bronze order rows.
if orders_inc_count > 0:
    # Create a window that keeps the latest order record for each order_id.
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the Silver order-cleaning pipeline. This block standardizes and deduplicates raw order records.
    orders_cleaned = (
        orders_inc
        # Standardize order_status to uppercase so values such as shipped and SHIPPED become consistent.
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn("order_status", F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status")))
        # Remove formatting characters from order_amount so it can be cast to a numeric type.
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
        .withColumn("order_amount", F.when(F.trim(F.col("order_amount")).isin("N/A", "NULL", "?", ""), None).otherwise(F.col("order_amount")))
        .withColumn("order_amount", F.col("order_amount").cast("double"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(order_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_cleaned,
        "datatocrunch_novacart_adb.silver_schema.orders_cleaned",
        "order_id"
    )

    # Apply Silver data-quality rules to the cleaned order records.
    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_by_orders_team",
            F.when(F.col("customer_id").isNull(), "verify_customer_id")
            .when(F.col("product_id").isNull(), "verify_product_id")
            .when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""), "verify_order_status")
            .when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), "verify_order_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), F.lit(True))
            .otherwise(F.lit(False))
        )
        .withColumn("order_date", F.to_date("created_at"))
        .withColumn("order_year", F.year("created_at"))
        .withColumn("order_month", F.month("created_at"))
        .withColumn("order_day", F.dayofmonth("created_at"))
        .withColumn("order_dow", F.date_format("created_at", "E"))
    )

    # Keep only valid order rows for the transformed Silver table.
    orders_good = orders_validated.filter(F.col("to_be_verified_by_orders_team") == "No Issues")

    # Send invalid order rows to the quarantine dataset for manual review.
    orders_bad = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team") != "No Issues")
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_good,
        "datatocrunch_novacart_adb.silver_schema.orders_transformed",
        "order_id"
    )

    # Append bad order rows to the quarantine table instead of losing them.
    orders_bad.write.format("delta").mode("append").saveAsTable(
        "datatocrunch_novacart_adb.silver_schema.orders_quarantine"
    )

    mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

    mx_run = (
        orders_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())

else:
    print("No new orders Bronze rows for Silver.")

    upsert_silver_control(
        "orders",
        None,
        last_orders_ingested_at,
        orders_inc_count
    )